# Startup Profit Regression
Pamela Vilchez | Applied graduate coursework

Can spending and state indicators predict profit in a 50-row teaching dataset? This streamlined portfolio adaptation follows the recovered multiple-regression assignment. It uses one fixed 70/30 split rather than the original repeated unseeded 70/30 and 90/10 experiments. Encoding is now fit inside a training-only pipeline; a mean baseline and error metrics are added. These revised results are not the original submission's results. No causal return on spending is estimated.

Supply an authorized `data/50_Startups.csv`. Raw records are not distributed.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

data = pd.read_csv('data/50_Startups.csv')
numeric = ['R&D Spend', 'Administration', 'Marketing Spend']
assert set(data.columns) == set(numeric + ['State', 'Profit'])
assert not data.isna().any().any(), 'Missing data requires review'
assert not data.duplicated().any(), 'Duplicates require review before splitting'
print('Dataset shape:', data.shape)
X_train, X_test, y_train, y_test = train_test_split(
    data[numeric + ['State']], data['Profit'], test_size=0.3, random_state=42)
assert set(X_train.index).isdisjoint(X_test.index)
prepare = ColumnTransformer([
    ('spending', 'passthrough', numeric),
    ('state', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), ['State'])])
model = Pipeline([('prepare', prepare), ('regression', LinearRegression())])
model.fit(X_train, y_train)
baseline = DummyRegressor(strategy='mean').fit(X_train, y_train)
rows=[]
for name, estimator in [('Training-mean baseline', baseline), ('Multiple linear regression', model)]:
    predicted = estimator.predict(X_test)
    assert np.isfinite(predicted).all()
    mse = mean_squared_error(y_test, predicted)
    rows.append({'Model':name, 'Test R2':r2_score(y_test,predicted),
                 'MAE':mean_absolute_error(y_test,predicted), 'MSE':mse, 'RMSE':np.sqrt(mse)})
print(pd.DataFrame(rows).to_string(index=False))
print('Training R2:', model.score(X_train,y_train))
summary={'rows':len(data),'train_rows':len(X_train),'test_rows':len(X_test),
         'seed':42,'training_r2':model.score(X_train,y_train),'evaluation':rows}
Path('results.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')

Dataset shape: (50, 5)


                     Model   Test R2          MAE          MSE         RMSE
    Training-mean baseline -0.004773 30301.287543 1.413716e+09 37599.420824
Multiple linear regression  0.939711  7395.433531 8.482696e+07  9210.154995
Training R2: 0.9511471170962384


## Interpretation
Compare test errors with the training-mean baseline to assess predictive information. R² is not a percentage of correct predictions. The test set has only 15 observations and results can be sensitive to the split. State indicators describe this sample and should not be interpreted as regional business performance in the population.

This exercise does not support reallocating marketing or R&D budgets: spending is observational, and unmeasured company characteristics may affect both spending and profit. Dataset provenance, currency and sampling are not established. A real planning application would need a larger verified sample, temporal validation and a causal design for budget recommendations.

## Coursework complement: OLS specification sensitivity
The additional named coursework notebook fits three OLS specifications to all 50 observations. The code below reproduces those formulas with corrected interpretation. This is an explanatory, in-sample comparison, separate from the fixed holdout evaluation above; it does not select or refit the predictive model using the test sample.

The source wording describes R² as accuracy. Here R² means the proportion of sample outcome variation explained by the fitted specification, not a percentage of correct predictions. Removing a predictor can change other coefficients, particularly when predictors are correlated.

In [2]:
import statsmodels.formula.api as smf
formulas = {
    'Full specification': "Profit ~ Q('R&D Spend') + Administration + Q('Marketing Spend') + C(State)",
    'Without marketing spend': "Profit ~ Q('R&D Spend') + Administration + C(State)",
    'Without R&D spend': "Profit ~ Q('Marketing Spend') + Administration + C(State)"}
ols_comparison = []
for name, formula in formulas.items():
    fit = smf.ols(formula, data=data).fit()
    ols_comparison.append({'Specification':name,'In-sample R2':fit.rsquared,
                           'Adjusted R2':fit.rsquared_adj,'Observations':int(fit.nobs)})
print(pd.DataFrame(ols_comparison).to_string(index=False))
print('R&D / marketing spend correlation:', data['R&D Spend'].corr(data['Marketing Spend']))
summary['ols_specification_comparison'] = ols_comparison
Path('results.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')

          Specification  In-sample R2  Adjusted R2  Observations
     Full specification      0.950752     0.945156            50
Without marketing spend      0.947980     0.943356            50
      Without R&D spend      0.613108     0.578717            50
R&D / marketing spend correlation: 0.7242481330538794


These models describe the observed sample. Differences in R² do not establish the causal importance or return on investment of any spending category. The original OLS output also flags a large condition number, which can reflect variable scale as well as predictor dependence. No causal budget recommendation follows from this comparison.